# TCSPC Fitting Notebook

This notebook imports TCSPC text files, converts channel number to time using the file-specific `Time calibration: ... ns/ch` line, normalizes the decay to a maximum of 1, and fits the normalized data using mono-, bi-, or tri-exponential models.

It also includes a **mono + bi comparison** mode that fits both models and overlays the fits on the same graph.

## Main workflow

1. Run all cells from top to bottom.
2. Upload one or more TCSPC `.txt`, `.csv`, `.dat`, or `.asc` files.
3. Choose the fit model.
4. Set the fit start and optional fit end.
5. Click **Fit uploaded files**.

Outputs are saved to your Downloads folder under `TCSPC_fits_normalized`.

In [ ]:
# If widgets do not display or buttons do not click, uncomment and run this line once.
# After installation, restart the kernel and refresh the browser tab.
# %pip install --upgrade ipywidgets jupyterlab_widgets widgetsnbextension

## 1. Imports and global settings

This cell imports the Python packages used throughout the notebook and defines the output folder.

In [ ]:
# Standard Python tools
import re
from pathlib import Path

# Numerical/data tools
import numpy as np
import pandas as pd

# Plotting and fitting
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Jupyter upload/button interface
from IPython.display import display, clear_output
import ipywidgets as widgets

# Save output files in the user's Downloads folder.
# The folder is created automatically if it does not already exist.
OUTPUT_DIR = Path.home() / "Downloads" / "TCSPC_fits_normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Background is estimated from the final fraction of the decay trace.
# 0.10 means the final 10% of points are used to estimate background.
BACKGROUND_FRACTION = 0.10

print(f"Output files will be saved to: {OUTPUT_DIR}")

## 2. Exponential decay models

These functions define the mathematical models used for fitting.

The amplitudes and background are fitted using **normalized counts**, so amplitudes should generally be around 0 to 1 instead of thousands of counts.

In [ ]:
def mono_exp(t, A, tau, bg):
    """
    Mono-exponential decay model.

    Parameters
    ----------
    t : array
        Time after peak, in ns.
    A : float
        Normalized amplitude.
    tau : float
        Lifetime, in ns.
    bg : float
        Normalized constant background.
    """
    return A * np.exp(-t / tau) + bg


def bi_exp(t, A1, tau1, A2, tau2, bg):
    """
    Bi-exponential decay model.

    The total decay is a sum of a fast and slow exponential component.
    """
    return A1 * np.exp(-t / tau1) + A2 * np.exp(-t / tau2) + bg


def tri_exp(t, A1, tau1, A2, tau2, A3, tau3, bg):
    """
    Tri-exponential decay model.

    Use this only when mono- and bi-exponential fits are clearly insufficient,
    because this model has many free parameters.
    """
    return (
        A1 * np.exp(-t / tau1)
        + A2 * np.exp(-t / tau2)
        + A3 * np.exp(-t / tau3)
        + bg
    )

## 3. TCSPC file loader

Your TCSPC files contain a channel column and a data/counts column. They also contain a line such as:

```text
Time calibration: 0.1097394ns/ch
```

This function reads that calibration value and converts channel number to time:

```python
time_ns = (channel - first_channel) * ns_per_channel
```

The fit function later shifts the peak to `t = 0`.

In [ ]:
def load_tcspc_file(file_content, filename=None):
    """
    Load a TCSPC text file and convert channel number to calibrated time.

    Expected file structure:

        Item name: Decay
        Real time: ...
        Live time: ...
        Time calibration: 0.1097394ns/ch

        Chan    Data
        1       4
        2       15
        ...

    Parameters
    ----------
    file_content : bytes, memoryview, or str
        Uploaded file contents.
    filename : str, optional
        File name used only for metadata/output labels.

    Returns
    -------
    df : pandas.DataFrame
        Columns: channel, time_ns, counts.
    metadata : dict
        Parsed metadata, including ns_per_channel.
    """

    # FileUpload may give a memoryview; convert it to bytes first.
    if isinstance(file_content, memoryview):
        file_content = file_content.tobytes()

    # Decode file text. errors="ignore" avoids crashing on odd characters.
    if isinstance(file_content, bytes):
        text = file_content.decode(errors="ignore")
    else:
        text = str(file_content)

    # Store useful information from the header.
    metadata = {
        "filename": filename,
        "item_name": None,
        "real_time_s": None,
        "live_time_s": None,
        "ns_per_channel": None,
    }

    # Parse item name, if present.
    item_match = re.search(r"Item\s+name\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if item_match:
        metadata["item_name"] = item_match.group(1).strip()

    # Parse real acquisition time, if present.
    real_match = re.search(
        r"Real\s+time\s*:\s*([+-]?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)",
        text,
        flags=re.IGNORECASE,
    )
    if real_match:
        metadata["real_time_s"] = float(real_match.group(1))

    # Parse live time, if present.
    live_match = re.search(
        r"Live\s+time\s*:\s*([+-]?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)",
        text,
        flags=re.IGNORECASE,
    )
    if live_match:
        metadata["live_time_s"] = float(live_match.group(1))

    # Parse time calibration, including scientific notation such as 5.486969E-02ns/ch.
    cal_match = re.search(
        r"Time\s+calibration\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][+-]?\d+)?)\s*ns\s*/\s*ch",
        text,
        flags=re.IGNORECASE,
    )

    if cal_match is None:
        raise ValueError("Could not find a valid 'Time calibration: ... ns/ch' line.")

    ns_per_channel = float(cal_match.group(1))
    metadata["ns_per_channel"] = ns_per_channel

    # Read the Chan/Data table.
    rows = []
    in_data_table = False

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        # Detect the table header.
        if re.match(r"^Chan\s+Data$", line, flags=re.IGNORECASE):
            in_data_table = True
            continue

        # After the header, parse numeric channel/count rows.
        if in_data_table:
            parts = re.split(r"[\t,; ]+", line)

            if len(parts) < 2:
                continue

            try:
                channel = int(float(parts[0]))
                counts = float(parts[1])
                rows.append((channel, counts))
            except ValueError:
                # Ignore any nonnumeric line after the header.
                continue

    if len(rows) == 0:
        raise ValueError("No Chan/Data table found after the header.")

    data = np.array(rows, dtype=float)
    channels = data[:, 0].astype(int)
    counts = data[:, 1]

    # Sort by channel just in case the file is not ordered.
    order = np.argsort(channels)
    channels = channels[order]
    counts = counts[order]

    # Convert channel number to time in ns.
    # Subtracting first_channel makes the first channel correspond to 0 ns.
    first_channel = channels.min()
    time_ns = (channels - first_channel) * ns_per_channel

    metadata["first_channel"] = int(first_channel)
    metadata["last_channel"] = int(channels.max())
    metadata["n_points"] = int(len(channels))

    df = pd.DataFrame({
        "channel": channels,
        "time_ns": time_ns,
        "counts": counts,
    })

    return df, metadata

## 4. Fit function

This function normalizes the decay to a maximum value of 1 **before fitting**.

Important details:

- Raw counts are preserved as `raw_counts`.
- Normalized counts are stored as `counts`.
- Poisson uncertainty is calculated from raw counts and then normalized by the same normalization factor.
- The peak is shifted to `t = 0`.
- The fit uses only the selected fit window, starting at `fit_start_ns`.

In [ ]:
def fit_tcspc(
    time_ns,
    counts,
    channels=None,
    metadata=None,
    model_name="bi-exponential",
    fit_start_ns=0.2,
    fit_end_ns=None,
):
    """
    Fit calibrated TCSPC decay data after normalizing counts to a maximum of 1.

    Parameters
    ----------
    time_ns : array
        Calibrated time axis in ns.
    counts : array
        Raw measured counts.
    channels : array, optional
        Channel numbers corresponding to the counts.
    metadata : dict, optional
        Metadata returned by load_tcspc_file().
    model_name : str
        "mono-exponential", "bi-exponential", or "tri-exponential".
    fit_start_ns : float
        Start of fit window after the peak, in ns.
    fit_end_ns : float or None
        End of fit window after the peak, in ns. None fits to the end.

    Returns
    -------
    result : dict
        Fit results, fitted parameters, normalized data, raw data, and metadata.
    """

    time_ns = np.asarray(time_ns, dtype=float)
    raw_counts = np.asarray(counts, dtype=float)

    if channels is not None:
        channels = np.asarray(channels)
    else:
        channels = np.arange(len(time_ns))

    # Remove invalid values.
    valid = np.isfinite(time_ns) & np.isfinite(raw_counts)
    time_ns = time_ns[valid]
    raw_counts = raw_counts[valid]
    channels = channels[valid]

    if len(time_ns) < 10:
        raise ValueError("Not enough valid data points to fit.")

    # Normalize y-axis before fitting.
    normalization_factor = np.max(raw_counts)

    if normalization_factor <= 0:
        raise ValueError("Cannot normalize data because maximum counts is zero.")

    counts = raw_counts / normalization_factor

    # Estimate normalized background from the final part of the trace.
    n_bg = max(5, int(len(counts) * BACKGROUND_FRACTION))
    bg_guess = np.median(counts[-n_bg:])

    # Find peak and shift time so peak is t = 0.
    peak_idx = np.argmax(counts)
    t0_ns = time_ns[peak_idx]
    peak_channel = channels[peak_idx]
    shifted_time = time_ns - t0_ns

    # Select the fit region after the peak.
    mask = shifted_time >= fit_start_ns

    if fit_end_ns is not None:
        mask &= shifted_time <= fit_end_ns

    t_fit = shifted_time[mask]
    y_fit = counts[mask]
    raw_y_fit = raw_counts[mask]
    channel_fit = channels[mask]
    raw_time_fit = time_ns[mask]

    if len(t_fit) < 10:
        raise ValueError("Fit window contains too few points. Adjust fit_start_ns or fit_end_ns.")

    # Poisson uncertainty in raw counts is sqrt(counts).
    # Since y was normalized, sigma must be normalized too.
    sigma = np.sqrt(np.maximum(raw_y_fit, 1)) / normalization_factor

    # Use normalized amplitudes for initial guesses.
    max_counts = np.max(y_fit)
    amp_guess = max(max_counts - bg_guess, 1e-6)

    # Choose the model and initial parameter guesses.
    if model_name == "mono-exponential":
        model = mono_exp
        p0 = [amp_guess, 10.0, bg_guess]
        bounds = ([0, 0, 0], [np.inf, np.inf, np.inf])
        param_names = ["A", "tau", "bg"]

    elif model_name == "bi-exponential":
        model = bi_exp
        p0 = [0.7 * amp_guess, 5.0, 0.3 * amp_guess, 50.0, bg_guess]
        bounds = ([0, 0, 0, 0, 0], [np.inf, np.inf, np.inf, np.inf, np.inf])
        param_names = ["A1", "tau1", "A2", "tau2", "bg"]

    elif model_name == "tri-exponential":
        model = tri_exp
        p0 = [
            0.5 * amp_guess, 2.0,
            0.3 * amp_guess, 20.0,
            0.2 * amp_guess, 100.0,
            bg_guess,
        ]
        bounds = (
            [0, 0, 0, 0, 0, 0, 0],
            [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf],
        )
        param_names = ["A1", "tau1", "A2", "tau2", "A3", "tau3", "bg"]

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    # Weighted nonlinear least-squares fit.
    popt, pcov = curve_fit(
        model,
        t_fit,
        y_fit,
        p0=p0,
        sigma=sigma,
        absolute_sigma=True,
        bounds=bounds,
        maxfev=50000,
    )

    # Parameter standard errors from covariance matrix.
    perr = np.sqrt(np.diag(pcov))

    # Calculate fitted curve and residuals in normalized-count units.
    y_model = model(t_fit, *popt)
    residuals = y_fit - y_model

    # Reduced chi-square.
    dof = max(1, len(y_fit) - len(popt))
    chi2_red = np.sum((residuals / sigma) ** 2) / dof

    result = {
        "metadata": metadata if metadata is not None else {},
        "channels": channels,
        "raw_time_ns": time_ns,
        "shifted_time_ns": shifted_time,

        # Normalized data used for fitting and plotting.
        "counts": counts,

        # Raw counts are kept so you can inspect/export original data.
        "raw_counts": raw_counts,
        "normalization_factor": normalization_factor,

        "channel_fit": channel_fit,
        "raw_time_fit_ns": raw_time_fit,
        "t_fit": t_fit,
        "y_fit": y_fit,
        "raw_y_fit": raw_y_fit,
        "y_model": y_model,
        "residuals": residuals,
        "sigma": sigma,
        "popt": popt,
        "perr": perr,
        "param_names": param_names,
        "chi2_red": chi2_red,
        "t0_original_ns": t0_ns,
        "peak_channel": peak_channel,
        "model_name": model_name,
    }

    return result

## 5. Plot and save a single-model fit

This function saves:

- A calibrated trace CSV
- A fit-region CSV
- A parameter CSV
- A fit plot PNG
- A residual plot PNG

The plotted y-axis is normalized counts.

In [ ]:
def plot_and_save_fit(filename, result):
    """
    Plot, display, and save a single TCSPC fit result.

    The data plotted here are normalized counts, because normalization happened
    inside fit_tcspc() before fitting.
    """
    base_name = Path(filename).stem
    metadata = result["metadata"]

    channels = result["channels"]
    raw_time_ns = result["raw_time_ns"]
    shifted_time_ns = result["shifted_time_ns"]
    counts = result["counts"]
    raw_counts = result["raw_counts"]

    t_fit = result["t_fit"]
    y_fit = result["y_fit"]
    y_model = result["y_model"]
    residuals = result["residuals"]

    ns_per_channel = metadata.get("ns_per_channel", np.nan)

    # Save full trace with raw and normalized counts.
    raw_df = pd.DataFrame({
        "channel": channels,
        "time_ns_raw": raw_time_ns,
        "time_ns_shifted": shifted_time_ns,
        "raw_counts": raw_counts,
        "normalized_counts": counts,
        "poisson_error_raw": np.sqrt(np.maximum(raw_counts, 1)),
        "poisson_error_normalized": np.sqrt(np.maximum(raw_counts, 1)) / result["normalization_factor"],
    })

    raw_csv_path = OUTPUT_DIR / f"{base_name}_calibrated_normalized_trace.csv"
    raw_df.to_csv(raw_csv_path, index=False)

    # Save only the data points that were included in the fit.
    fit_df = pd.DataFrame({
        "channel": result["channel_fit"],
        "time_ns_raw": result["raw_time_fit_ns"],
        "time_ns_shifted": t_fit,
        "raw_counts": result["raw_y_fit"],
        "normalized_counts": y_fit,
        "fit_normalized": y_model,
        "residual_normalized": residuals,
        "poisson_error_normalized": result["sigma"],
    })

    safe_model_name = result["model_name"].replace(" ", "_")
    fit_csv_path = OUTPUT_DIR / f"{base_name}_{safe_model_name}_fit.csv"
    fit_df.to_csv(fit_csv_path, index=False)

    # Save fitted parameters.
    params = {
        "filename": filename,
        "item_name": metadata.get("item_name"),
        "model": result["model_name"],
        "ns_per_channel": ns_per_channel,
        "first_channel": metadata.get("first_channel"),
        "last_channel": metadata.get("last_channel"),
        "n_points": metadata.get("n_points"),
        "real_time_s": metadata.get("real_time_s"),
        "live_time_s": metadata.get("live_time_s"),
        "normalization_factor_raw_counts": result["normalization_factor"],
        "peak_channel": result["peak_channel"],
        "t0_original_ns": result["t0_original_ns"],
        "reduced_chi_square": result["chi2_red"],
    }

    for name, val, err in zip(result["param_names"], result["popt"], result["perr"]):
        params[name] = val
        params[f"{name}_error"] = err

    params_df = pd.DataFrame([params])
    params_path = OUTPUT_DIR / f"{base_name}_{safe_model_name}_fit_parameters.csv"
    params_df.to_csv(params_path, index=False)

    # Main fit plot.
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(shifted_time_ns, counts, s=10, label="Normalized data")
    ax.plot(t_fit, y_model, color="red", linewidth=2, label=result["model_name"])

    ax.set_xlabel("Time after peak (ns)")
    ax.set_ylabel("Normalized counts")
    ax.set_title(f"{base_name} — {result['model_name']}")
    ax.set_yscale("log", nonpositive="clip")
    ax.grid(True, alpha=0.3)
    ax.legend()

    text = (
        f"cal = {ns_per_channel:.6g} ns/ch\n"
        f"peak ch = {result['peak_channel']}\n"
        f"norm = {result['normalization_factor']:.4g} raw counts\n"
        f"reduced chi^2 = {result['chi2_red']:.3g}\n"
    )

    for name, val, err in zip(result["param_names"], result["popt"], result["perr"]):
        text += f"{name} = {val:.4g} +/- {err:.2g}\n"

    ax.text(
        0.98,
        0.98,
        text,
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.8),
    )

    png_path = OUTPUT_DIR / f"{base_name}_{safe_model_name}_fit.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Residual plot.
    fig, ax = plt.subplots(figsize=(7, 3))

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.scatter(t_fit, residuals, s=10)

    ax.set_xlabel("Time after peak (ns)")
    ax.set_ylabel("Residuals, normalized counts")
    ax.set_title(f"{base_name} residuals")
    ax.grid(True, alpha=0.3)

    residual_path = OUTPUT_DIR / f"{base_name}_{safe_model_name}_residuals.png"
    plt.savefig(residual_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:")
    print(f"  {raw_csv_path}")
    print(f"  {fit_csv_path}")
    print(f"  {params_path}")
    print(f"  {png_path}")
    print(f"  {residual_path}")

    return params_df

## 6. Plot and save mono vs bi comparison

This function fits the same normalized data to both mono- and bi-exponential models, then overlays them on the same graph.

Color convention:

- Mono fit = black
- Bi fit = red

In [ ]:
def plot_and_save_mono_bi_comparison(filename, mono_result, bi_result):
    """
    Plot and save a mono-exponential vs bi-exponential comparison.

    Both fits are performed on normalized data. The same normalized decay is
    used for both models.
    """
    base_name = Path(filename).stem
    metadata = mono_result["metadata"]

    shifted_time_ns = mono_result["shifted_time_ns"]
    counts = mono_result["counts"]

    # Main comparison plot.
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(shifted_time_ns, counts, s=10, label="Normalized data")

    ax.plot(
        mono_result["t_fit"],
        mono_result["y_model"],
        color="black",
        linewidth=2,
        label=f"Mono fit, chi^2 = {mono_result['chi2_red']:.3g}",
    )

    ax.plot(
        bi_result["t_fit"],
        bi_result["y_model"],
        color="red",
        linewidth=2,
        label=f"Bi fit, chi^2 = {bi_result['chi2_red']:.3g}",
    )

    ax.set_xlabel("Time after peak (ns)")
    ax.set_ylabel("Normalized counts")
    ax.set_title(f"{base_name} — mono vs bi comparison")
    ax.set_yscale("log", nonpositive="clip")
    ax.grid(True, alpha=0.3)
    ax.legend()

    # Build a text box containing the key fit parameters.
    text = (
        f"Mono reduced chi^2 = {mono_result['chi2_red']:.3g}\n"
        f"Bi reduced chi^2 = {bi_result['chi2_red']:.3g}\n"
        f"norm = {mono_result['normalization_factor']:.4g} raw counts\n\n"
    )

    text += "Mono:\n"
    for name, val, err in zip(mono_result["param_names"], mono_result["popt"], mono_result["perr"]):
        text += f"{name} = {val:.4g} +/- {err:.2g}\n"

    text += "\nBi:\n"
    for name, val, err in zip(bi_result["param_names"], bi_result["popt"], bi_result["perr"]):
        text += f"{name} = {val:.4g} +/- {err:.2g}\n"

    ax.text(
        0.98,
        0.98,
        text,
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.8),
    )

    png_path = OUTPUT_DIR / f"{base_name}_mono_bi_comparison.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Residual comparison plot.
    fig, ax = plt.subplots(figsize=(7, 3))

    ax.axhline(0, linestyle="--", linewidth=1)

    ax.scatter(
        mono_result["t_fit"],
        mono_result["residuals"],
        color="black",
        s=10,
        label="Mono residuals",
    )

    ax.scatter(
        bi_result["t_fit"],
        bi_result["residuals"],
        color="red",
        s=10,
        label="Bi residuals",
    )

    ax.set_xlabel("Time after peak (ns)")
    ax.set_ylabel("Residuals, normalized counts")
    ax.set_title(f"{base_name} — residual comparison")
    ax.grid(True, alpha=0.3)
    ax.legend()

    residual_path = OUTPUT_DIR / f"{base_name}_mono_bi_residuals.png"
    plt.savefig(residual_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Save comparison parameters in one CSV with one row for mono and one row for bi.
    rows = []

    for result in [mono_result, bi_result]:
        row = {
            "filename": filename,
            "model": result["model_name"],
            "reduced_chi_square": result["chi2_red"],
            "normalization_factor_raw_counts": result["normalization_factor"],
            "peak_channel": result["peak_channel"],
            "t0_original_ns": result["t0_original_ns"],
            "ns_per_channel": metadata.get("ns_per_channel"),
        }

        for name, val, err in zip(result["param_names"], result["popt"], result["perr"]):
            row[name] = val
            row[f"{name}_error"] = err

        rows.append(row)

    comparison_df = pd.DataFrame(rows)
    comparison_path = OUTPUT_DIR / f"{base_name}_mono_bi_comparison_parameters.csv"
    comparison_df.to_csv(comparison_path, index=False)

    print("Saved comparison:")
    print(f"  {png_path}")
    print(f"  {residual_path}")
    print(f"  {comparison_path}")

    return comparison_df

## 7. Upload interface and run button

Run this cell last. It displays the upload widget and fitting controls.

The model dropdown includes:

- `mono-exponential`
- `bi-exponential`
- `tri-exponential`
- `mono + bi comparison`

In [ ]:
# File upload widget. Multiple files can be uploaded at once.
uploader = widgets.FileUpload(
    accept=".txt,.csv,.dat,.asc",
    multiple=True
)

# Model selection dropdown.
model_dropdown = widgets.Dropdown(
    options=[
        "mono-exponential",
        "bi-exponential",
        "tri-exponential",
        "mono + bi comparison",
    ],
    value="bi-exponential",
    description="Model:"
)

# Fit start is relative to the peak position after time shifting.
# Increase this if the early-time prompt/IRF region distorts the fit.
fit_start_box = widgets.FloatText(
    value=0.2,
    description="Fit start ns:"
)

# Fit end is also relative to the peak position.
# A value of -1 means fit until the end of the trace.
fit_end_box = widgets.FloatText(
    value=-1,
    description="Fit end ns:"
)

run_button = widgets.Button(
    description="Fit uploaded files",
    button_style="success"
)

output = widgets.Output()


def get_uploaded_files(upload_widget):
    """
    Extract uploaded file names and contents from the FileUpload widget.

    This handles both ipywidgets 7 and ipywidgets 8 formats.
    """
    uploaded = upload_widget.value

    if uploaded is None or len(uploaded) == 0:
        return []

    files = []

    # ipywidgets 7 format: dictionary of file information.
    if isinstance(uploaded, dict):
        for filename, fileinfo in uploaded.items():
            files.append((filename, fileinfo["content"]))

    # ipywidgets 8 format: list/tuple of file dictionaries.
    elif isinstance(uploaded, (list, tuple)):
        for fileinfo in uploaded:
            filename = fileinfo.get("name", "uploaded_tcspc_file.txt")
            content = fileinfo.get("content")
            files.append((filename, content))

    else:
        raise ValueError(f"Unknown upload format: {type(uploaded)}")

    return files


def run_single_model_fit(filename, trace_df, metadata, fit_end_ns):
    """
    Run one selected model and save/plot the result.
    """
    result = fit_tcspc(
        time_ns=trace_df["time_ns"].values,
        counts=trace_df["counts"].values,
        channels=trace_df["channel"].values,
        metadata=metadata,
        model_name=model_dropdown.value,
        fit_start_ns=fit_start_box.value,
        fit_end_ns=fit_end_ns,
    )

    params_df = plot_and_save_fit(filename, result)
    return params_df


def run_mono_bi_comparison(filename, trace_df, metadata, fit_end_ns):
    """
    Run both mono- and bi-exponential fits on the same file.
    """
    mono_result = fit_tcspc(
        time_ns=trace_df["time_ns"].values,
        counts=trace_df["counts"].values,
        channels=trace_df["channel"].values,
        metadata=metadata,
        model_name="mono-exponential",
        fit_start_ns=fit_start_box.value,
        fit_end_ns=fit_end_ns,
    )

    bi_result = fit_tcspc(
        time_ns=trace_df["time_ns"].values,
        counts=trace_df["counts"].values,
        channels=trace_df["channel"].values,
        metadata=metadata,
        model_name="bi-exponential",
        fit_start_ns=fit_start_box.value,
        fit_end_ns=fit_end_ns,
    )

    params_df = plot_and_save_mono_bi_comparison(
        filename,
        mono_result,
        bi_result,
    )

    return params_df


def run_fits(button):
    """
    Main button callback.

    This function runs when the user clicks the Fit uploaded files button.
    """
    with output:
        clear_output(wait=True)

        print("Button clicked. Starting TCSPC fitting...")

        uploaded_files = get_uploaded_files(uploader)

        if len(uploaded_files) == 0:
            print("No files uploaded yet.")
            print("Upload one or more TCSPC files, then press the button again.")
            return

        all_params = []

        fit_end_ns = fit_end_box.value
        if fit_end_ns < 0:
            fit_end_ns = None

        print(f"Number of uploaded files: {len(uploaded_files)}")
        print(f"Selected model: {model_dropdown.value}")
        print(f"Fit start: {fit_start_box.value} ns")
        print(f"Fit end: {fit_end_ns} ns")
        print("")

        for filename, content in uploaded_files:
            print(f"Processing {filename}...")

            try:
                # Load file and convert channel to calibrated time.
                trace_df, metadata = load_tcspc_file(content, filename=filename)

                print(f"  Time calibration: {metadata['ns_per_channel']} ns/ch")
                print(f"  Channels: {metadata['first_channel']} to {metadata['last_channel']}")
                print(f"  Points: {metadata['n_points']}")

                if model_dropdown.value == "mono + bi comparison":
                    params_df = run_mono_bi_comparison(
                        filename,
                        trace_df,
                        metadata,
                        fit_end_ns,
                    )
                else:
                    params_df = run_single_model_fit(
                        filename,
                        trace_df,
                        metadata,
                        fit_end_ns,
                    )

                all_params.append(params_df)

                print(f"Finished {filename}")
                print("")

            except Exception as e:
                print(f"Failed to process {filename}")
                print(f"Error: {e}")
                print("")

        if all_params:
            summary_df = pd.concat(all_params, ignore_index=True)
            summary_path = OUTPUT_DIR / "TCSPC_fit_summary.csv"
            summary_df.to_csv(summary_path, index=False)

            print("All done.")
            print(f"Summary saved to: {summary_path}")

            display(summary_df)


# Attach the callback to the button.
run_button.on_click(run_fits)

# Display the full user interface.
display(
    widgets.VBox([
        uploader,
        model_dropdown,
        fit_start_box,
        fit_end_box,
        run_button,
        output,
    ])
)